# 02 - Train the context encoder (homograph disambiguation)

This is the cheap, high-value stage: about 20 minutes on a 4090. It is also
where homograph accuracy actually comes from, so iterate here before spending
money on the acoustic model.

The model is about 6M parameters. For every word it predicts which of that
word's discovered readings applies in this context.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp0_small.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
# Printed because the CATT env exists alongside this one: notebook 01 calls the
# CATT interpreter for one stage, and a shell that activated it would otherwise
# hijack a bare `python`. Every stage below runs {sys.executable}, so it follows
# the kernel rather than the shell.
print("python :", sys.executable)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

## Start TensorBoard

Watch `eval/code_acc`. That is held-out homograph accuracy, the number that
matters, and it has to be read **against the majority baseline** from notebook
01, not against zero. A model that always guesses the commonest reading already
scores the baseline while having learned nothing.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $cfg.paths.tb_dir --port 6006 --bind_all

## Train

In [ ]:
!{sys.executable} scripts/train_context.py --config $CONFIG

## Evaluate what it learned

The two `علم` sentences must get *different* codes. If they do not,
disambiguation is not working and the acoustic model will inherit the failure.

In [ ]:
import json
from adaptts.infer.pipeline import AdapTTS

tts = AdapTTS.from_checkpoints(CONFIG, device="cpu", load_codec=False)
probes = json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]

for p in probes:
    plan = tts.analyze(p["text"])
    hard = [f"{w.word}=code{w.code}({w.confidence:.2f})" for w in plan.hard_words]
    print()
    print(p["tag"])
    print("  text      :", p["text"])
    print("  expected  :", p["expected"])
    print(f"  difficulty: {plan.sentence_difficulty:.3f} -> depth {plan.depth}")
    print("  decisions :", ", ".join(hard) if hard else "no ambiguous words")

In [ ]:
# The critical comparison: one word, two contexts, two readings.
a = tts.analyze("انا شوفت علم مصر بيرفرف")             # flag
b = tts.analyze("علم الفيزيا من اهم العلوم البشرية")   # science
ca = [w.code for w in a.hard_words if w.word == "علم"]
cb = [w.code for w in b.hard_words if w.word == "علم"]
print("flag context    -> code", ca)
print("science context -> code", cb)
print()
print("DISAMBIGUATION WORKS" if ca and cb and ca != cb
      else "NOT disambiguating: investigate before training the acoustic model")

## The gate: did it beat the majority baseline?

The probe check above is necessary but not sufficient. Two different codes on
two sentences is consistent with a model that has genuinely learned context, and
also with one that memorised a single split. This cell measures held-out
accuracy against the baseline over every ambiguous occurrence.

**Do not start notebook 03 until this passes.** The acoustic model inherits
these labels; if the disambiguator is at the prior, the expensive run produces a
system that reads homographs by frequency, which is what the first $15 bought.

In [ ]:
import json, collections
from adaptts.text.diacritics import ReadingLexicon

lex = ReadingLexicon.load(cfg.paths.reading_lexicon_path)
rows = [json.loads(l) for l in open(cfg.paths.diacritized_path, encoding="utf-8")]
manifest = {json.loads(l)["uid"]: json.loads(l) for l in
            open(cfg.paths.manifest_path, encoding="utf-8")}

# Held-out splits only: training accuracy proves nothing about generalisation.
# The manifest names them "dev" and "test".
eval_uids = {u for u, r in manifest.items() if r.get("split") in ("dev", "test")}
print(f"held-out utterances: {len(eval_uids)}")

# Group the gold labels by sentence so each sentence is analyzed once rather
# than once per ambiguous word in it.
gold_by_text = collections.defaultdict(list)   # text -> [(word_index, word, code)]
for r in rows:
    if r["uid"] not in eval_uids:
        continue
    plain, diac = r["text"].split(), r["diacritized"].split()
    if len(plain) != len(diac):
        continue
    for i, (w, d) in enumerate(zip(plain, diac)):
        e = lex.entries.get(w)
        if e is None or e.n_codes < 2:
            continue
        c = e.code_of(d)
        if c >= 0:
            gold_by_text[r["text"]].append((i, w, c))

gold = [(t, i, w, c) for t, items in gold_by_text.items() for i, w, c in items]
print(f"held-out ambiguous occurrences: {len(gold)} "
      f"across {len(gold_by_text)} sentences")

# Majority baseline on exactly this set.
majority = {w: max(range(e.n_codes), key=lambda k: e.counts[k])
            for w, e in lex.entries.items() if e.n_codes > 1}
base_hits = sum(1 for _, _, w, c in gold if majority.get(w) == c)

# Model predictions, one analyze() per sentence.
from tqdm.auto import tqdm

hits = 0
by_word = collections.defaultdict(lambda: [0, 0])
for text, items in tqdm(gold_by_text.items(), desc="scoring", unit="sent"):
    plan = tts.analyze(text)
    # Match on word index, not on the word string: a sentence can repeat an
    # ambiguous word with two different readings (دول ... دول), and matching by
    # string alone would score the wrong occurrence.
    pred_by_idx = {hw.index: hw.code for hw in plan.hard_words}
    for idx, w, c in items:
        ok = pred_by_idx.get(idx) == c
        hits += int(ok)
        by_word[w][0] += int(ok)
        by_word[w][1] += 1

n = max(len(gold), 1)
acc, base = hits / n, base_hits / n
print()
print(f"majority baseline : {base:.3f}")
print(f"model accuracy    : {acc:.3f}")
print(f"margin            : {acc - base:+.3f}")
print()
if acc > base + 0.05:
    print("PASS - the model is using context. Continue to notebook 03.")
elif acc > base:
    print("MARGINAL - above the prior but within noise. More data or more")
    print("homograph contrast is needed before paying for the acoustic run.")
else:
    print("FAIL - at or below the prior. The model has learned nothing beyond")
    print("word frequency. Do NOT train the acoustic model yet.")

print()
print(f"{'word':<14}{'acc':>6}{'n':>6}")
for w, (h, t) in sorted(by_word.items(), key=lambda kv: -kv[1][1])[:20]:
    print(f"{w:<14}{h/max(t,1):>6.2f}{t:>6}")

In [ ]:
# The multi-homograph stress sentence from the brief.
plan = tts.analyze(
    "انا كنت مصر على ان مصر عندها امكانيات و موارد تخليها تتفوق على دول من اللي شايفين نفسهم دول"
)
print(plan)

## Inference speed on CPU

The context encoder must be negligible next to the acoustic model.

In [ ]:
import time

txt = "انا كنت مصر على ان مصر عندها امكانيات تخليها تتفوق على دول"
for _ in range(3):
    tts.analyze(txt)                      # warm up
t0 = time.perf_counter()
for _ in range(50):
    tts.analyze(txt)
print(f"context encoder: {(time.perf_counter() - t0) / 50 * 1000:.2f} ms per sentence on CPU")

If accuracy looks good, continue to **03_train_acoustic.ipynb**.